In [3]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
import joblib
import os

In [4]:
#dataset loading
df = pd.read_csv("data/heart_2020_cleaned.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (319795, 18)

Columns: ['HeartDisease', 'BMI', 'Smoking', 'AlcoholDrinking', 'Stroke', 'PhysicalHealth', 'MentalHealth', 'DiffWalking', 'Sex', 'AgeCategory', 'Race', 'Diabetic', 'PhysicalActivity', 'GenHealth', 'SleepTime', 'Asthma', 'KidneyDisease', 'SkinCancer']

First 5 rows:


,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,No,16.60,Yes,No,No,3.0,30.0,No,Female,55-59,White,Yes,Yes,Very good,5.0,Yes,No,Yes
1,No,20.34,No,No,Yes,0.0,0.0,No,Female,80 or older,White,No,Yes,Very good,7.0,No,No,No
2,No,26.58,Yes,No,No,20.0,30.0,No,Male,65-69,White,Yes,Yes,Fair,8.0,Yes,No,No
3,No,24.21,No,No,No,0.0,0.0,No,Female,75-79,White,No,No,Good,6.0,No,No,Yes
4,No,23.71,No,No,No,28.0,0.0,Yes,Female,40-44,White,No,Yes,Very good,8.0,No,No,No


In [5]:
#data
print("Target value counts:")
print(df["HeartDisease"].value_counts())
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)

Target value counts:
HeartDisease
No     292422
Yes     27373
Name: count, dtype: int64

Missing values:
HeartDisease        0
BMI                 0
Smoking             0
AlcoholDrinking     0
Stroke              0
PhysicalHealth      0
MentalHealth        0
DiffWalking         0
Sex                 0
AgeCategory         0
Race                0
Diabetic            0
PhysicalActivity    0
GenHealth           0
SleepTime           0
Asthma              0
KidneyDisease       0
SkinCancer          0
dtype: int64

Data types:
HeartDisease         object
BMI                 float64
Smoking              object
AlcoholDrinking      object
Stroke               object
PhysicalHealth      float64
MentalHealth        float64
DiffWalking          object
Sex                  object
AgeCategory          object
Race                 object
Diabetic             object
PhysicalActivity     object
GenHealth            object
SleepTime           float64
Asthma               object
KidneyDisease        obje

In [6]:
#encoding
categorical_cols = [
    "HeartDisease", "Smoking", "AlcoholDrinking", "Stroke",
    "DiffWalking", "Sex", "AgeCategory", "Race", "Diabetic",
    "PhysicalActivity", "GenHealth", "Asthma", "KidneyDisease", "SkinCancer"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

HeartDisease: {'No': np.int64(0), 'Yes': np.int64(1)}
Smoking: {'No': np.int64(0), 'Yes': np.int64(1)}
AlcoholDrinking: {'No': np.int64(0), 'Yes': np.int64(1)}
Stroke: {'No': np.int64(0), 'Yes': np.int64(1)}
DiffWalking: {'No': np.int64(0), 'Yes': np.int64(1)}
Sex: {'Female': np.int64(0), 'Male': np.int64(1)}
AgeCategory: {'18-24': np.int64(0), '25-29': np.int64(1), '30-34': np.int64(2), '35-39': np.int64(3), '40-44': np.int64(4), '45-49': np.int64(5), '50-54': np.int64(6), '55-59': np.int64(7), '60-64': np.int64(8), '65-69': np.int64(9), '70-74': np.int64(10), '75-79': np.int64(11), '80 or older': np.int64(12)}
Race: {'American Indian/Alaskan Native': np.int64(0), 'Asian': np.int64(1), 'Black': np.int64(2), 'Hispanic': np.int64(3), 'Other': np.int64(4), 'White': np.int64(5)}
Diabetic: {'No': np.int64(0), 'No, borderline diabetes': np.int64(1), 'Yes': np.int64(2), 'Yes (during pregnancy)': np.int64(3)}
PhysicalActivity: {'No': np.int64(0), 'Yes': np.int64(1)}
GenHealth: {'Excellent': n

In [7]:
#separate features & target
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

print("Feature columns:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class distribution:\n", y.value_counts())

Feature columns: ['BMI', 'Smoking', 'AlcoholDrinking', 'Stroke', 'PhysicalHealth', 'MentalHealth', 'DiffWalking', 'Sex', 'AgeCategory', 'Race', 'Diabetic', 'PhysicalActivity', 'GenHealth', 'SleepTime', 'Asthma', 'KidneyDisease', 'SkinCancer']
X shape: (319795, 17)
y shape: (319795,)
Class distribution:
 HeartDisease
0    292422
1     27373
Name: count, dtype: int64


In [8]:
#scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling complete.")
print("Mean of first feature:", scaler.mean_[0])

Scaling complete.
Mean of first feature: 28.325398520927465


In [9]:
#train test splitting
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 255836
Testing samples: 63959


In [10]:
#training the model
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


In [11]:
#check accuracy
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("=" * 45)
print(f"Accuracy:        {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score:   {roc_auc_score(y_test, y_pred_prob):.4f}")
print("=" * 45)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Heart Disease"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:        0.7405
ROC-AUC Score:   0.8263

Classification Report:
               precision    recall  f1-score   support

   No Disease       0.97      0.74      0.84     58484
Heart Disease       0.21      0.76      0.34      5475

     accuracy                           0.74     63959
    macro avg       0.59      0.75      0.59     63959
 weighted avg       0.91      0.74      0.80     63959


Confusion Matrix:
[[43184 15300]
 [ 1295  4180]]


In [12]:
#saving the model
os.makedirs("model", exist_ok=True)

joblib.dump(model,           "model/cardiosense_model.pkl")
joblib.dump(scaler,          "model/scaler.pkl")
joblib.dump(encoders,        "model/encoders.pkl")
joblib.dump(X.columns.tolist(), "model/feature_columns.pkl")

print("Saved:")
print("  model/cardiosense_model.pkl")
print("  model/scaler.pkl")
print("  model/encoders.pkl")
print("  model/feature_columns.pkl")

Saved:
  model/cardiosense_model.pkl
  model/scaler.pkl
  model/encoders.pkl
  model/feature_columns.pkl


In [13]:
#SAMPLE 
loaded_model  = joblib.load("model/cardiosense_model.pkl")
loaded_scaler = joblib.load("model/scaler.pkl")
loaded_cols   = joblib.load("model/feature_columns.pkl")

sample = pd.DataFrame([{
    "BMI": 28.5,
    "Smoking": 1,
    "AlcoholDrinking": 0,
    "Stroke": 0,
    "PhysicalHealth": 5,
    "MentalHealth": 3,
    "DiffWalking": 0,
    "Sex": 1,
    "AgeCategory": 7,
    "Race": 5,
    "Diabetic": 1,
    "PhysicalActivity": 1,
    "GenHealth": 2,
    "SleepTime": 7,
    "Asthma": 0,
    "KidneyDisease": 0,
    "SkinCancer": 0
}])[loaded_cols]

sample_scaled = loaded_scaler.transform(sample)
prob  = loaded_model.predict_proba(sample_scaled)[0][1]
label = "High Risk" if prob >= 0.6 else "Moderate Risk" if prob >= 0.3 else "Low Risk"

print(f"Risk Probability : {prob:.4f}  ({prob*100:.1f}%)")
print(f"Risk Label       : {label}")

Risk Probability : 0.5296  (53.0%)
Risk Label       : Moderate Risk
